In [1]:
import pandas as pd
from pathlib import Path
import numpy as np


In [2]:
carpeta_datos = Path(r"D:\ProyectoAnalisisElectrico\PotencialesClientes")

df_centros_pct = pd.read_parquet(carpeta_datos / "2505_2604_centroids_porcentual_actives.parquet")
df = pd.read_parquet(carpeta_datos / "2505_2604_clustered_porcentual_actives.parquet")


In [3]:
df.columns

Index(['clave', 'RUT_CLIENTE', 'REGION_CLIENTE', 'macrozona', 'Zona', 'Hora',
       'medida_count', 'medida_min', 'CLIENTE', 'CLIENTE_log', 'n_clientes',
       'TIPO', 'NOMBRE_ESTABLECIMIENTO', 'COMBUSTIBLE_PRIMARIO', 'SECTOR',
       'SUBSECTOR', 'RUBRO', 'DEMANDA_CALOR_MWH_sum', 'DEMANDA_CALOR_MWH_mean',
       'DEMANDA_CALOR_MWH_std', 'DEMANDA_CALOR_MWH_max',
       'DEMANDA_CALOR_MWH_min', 'RUT_PROVEEDOR', 'RUT_PROVEEDOR_log',
       'n_rut_proveedores', 'PROVEEDOR', 'PROVEEDOR_log', 'n_proveedores',
       'nombre_barra', 'nombre_barra_log', 'n_nombres_barra', 'tension',
       'tension_log', 'n_tensiones', 'Nombre_Corto', 'Nombre_Corto_log',
       'n_nombres_cortos', 'periodo_last', 'periodos_log', 'meses_operados',
       'medida_mean', 'medida_std', 'CMg[CLP/KWh]_mean', 'CMg[CLP/KWh]_std',
       'CMg[CLP/KWh]_count', 'valorizado_CLP_mean', 'valorizado_CLP_std',
       'valorizado_CLP_count', 'medida_total', 'medida_porcentual',
       'clave_unica', 'id_cluster', 'metodo_cl

In [4]:
# ==========================================
# 0. LIMPIEZA PREVENTIVA Y APERTURA SECTORIAL
# ==========================================
df['SECTOR'] = df['SECTOR'].fillna('Sin_Sector')
df['SUBSECTOR'] = df['SUBSECTOR'].fillna('Sin_Subsector')
df['medida_total'] = df['medida_total'].fillna(0)

# ABRIMOS EL SECTOR INDUSTRIAL
df['SECTOR'] = np.where(df['SECTOR'].str.lower() == 'industrial', df['SUBSECTOR'], df['SECTOR'])


# ==========================================
# 1. TABLA BASE (Conteos y Listas)
# ==========================================
df_resultados = df.groupby(['macrozona', 'id_cluster']).agg(
    clientes_del_cluster=('RUT_CLIENTE', 'nunique'),
    lista_ruts=('RUT_CLIENTE', 'unique'),
    lista_nombres=('CLIENTE', 'unique') 
).reset_index()

ruts_por_macro = df.groupby('macrozona')['RUT_CLIENTE'].nunique()
df_resultados['clientes_totales_macrozona'] = df_resultados['macrozona'].map(ruts_por_macro)


# ==========================================
# 2. FUNCIÓN MAESTRA DE PROPORCIONES Y TOTALES
# ==========================================
def calcular_proporciones_y_totales(df_base, columnas_agrupacion, tipo_calculo, nombre_col_prop, nombre_col_total):
    """
    Calcula la proporción de sectores dependiendo del enfoque (RUT, Claves, Energía o Calor),
    pero también rescata el total absoluto para no perder la magnitud de los datos.
    """
    if tipo_calculo == 'RUT':
        df_filtro = df_base[columnas_agrupacion + ['RUT_CLIENTE', 'SECTOR']].drop_duplicates()
        pivot = pd.pivot_table(df_filtro, index=columnas_agrupacion, columns='SECTOR', values='RUT_CLIENTE', aggfunc='count', fill_value=0)
        
    elif tipo_calculo == 'CLAVES':
        df_filtro = df_base[columnas_agrupacion + ['clave_unica', 'SECTOR']].drop_duplicates(subset=['clave_unica'])
        pivot = pd.pivot_table(df_filtro, index=columnas_agrupacion, columns='SECTOR', aggfunc='size', fill_value=0)
        
    elif tipo_calculo == 'ENERGIA':
        # Traemos también 'medida_count' para poder calcular el total anual
        cols_energia = columnas_agrupacion + ['clave_unica', 'SECTOR', 'medida_total', 'medida_count']
        
        # Usamos .copy() para evitar advertencias de Pandas
        df_filtro = df_base[cols_energia].drop_duplicates(subset=['clave_unica']).copy()
        
        # Anualizamos la energía y dividimos por 1000 para pasar de kWh a MWh
        df_filtro['energia_anual_real_mwh'] = (df_filtro['medida_total'] * df_filtro['medida_count']) / 1000
        
        # Pivotamos sumando la energía anualizada ya en MWh
        pivot = pd.pivot_table(df_filtro, index=columnas_agrupacion, columns='SECTOR', values='energia_anual_real_mwh', aggfunc='sum', fill_value=0)
        
    elif tipo_calculo == 'CALOR':
        columnas_calor = columnas_agrupacion + ['RUT_CLIENTE', 'REGION_CLIENTE', 'SECTOR', 'DEMANDA_CALOR_MWH_sum']
        subset_unicos = columnas_agrupacion + ['RUT_CLIENTE', 'REGION_CLIENTE']
        
        df_filtro = df_base[columnas_calor].drop_duplicates(subset=subset_unicos)
        pivot = pd.pivot_table(df_filtro, index=columnas_agrupacion, columns='SECTOR', values='DEMANDA_CALOR_MWH_sum', aggfunc='sum', fill_value=0)
    
        
    # Calcular el total absoluto de la fila
    totales = pivot.sum(axis=1).fillna(0)
    
    # Transformar a proporción (0 a 1) usando el total calculado y formatear a texto
    prop = pivot.div(totales, axis=0).fillna(0).round(4)
    texto = prop.apply(lambda row: "::".join([f"{sector}_{val:g}" for sector, val in row.items()]), axis=1)
    
    # Retornar ambas variables en un DataFrame
    df_salida = pd.DataFrame({
        nombre_col_prop: texto,
        nombre_col_total: totales
    }).reset_index()
    
    return df_salida


# ==========================================
# 3. APLICAR FUNCIÓN AL NIVEL CLUSTER
# ==========================================
cols_cluster = ['macrozona', 'id_cluster']

df_resultados = df_resultados.merge(calcular_proporciones_y_totales(df, cols_cluster, 'RUT', 'sect_rut_cluster', 'total_rut_cluster'), on=cols_cluster, how='left')
df_resultados = df_resultados.merge(calcular_proporciones_y_totales(df, cols_cluster, 'CLAVES', 'sect_claves_cluster', 'total_claves_cluster'), on=cols_cluster, how='left')
# Actualizamos el nombre de la columna a _mwh
df_resultados = df_resultados.merge(calcular_proporciones_y_totales(df, cols_cluster, 'ENERGIA', 'sect_energia_cluster', 'total_energia_cluster_mwh'), on=cols_cluster, how='left')
df_resultados = df_resultados.merge(calcular_proporciones_y_totales(df, cols_cluster, 'CALOR', 'sect_calor_cluster', 'total_calor_cluster_mwh'), on=cols_cluster, how='left')

# ==========================================
# 4. APLICAR FUNCIÓN AL NIVEL MACROZONA
# ==========================================
cols_macro = ['macrozona']

df_resultados = df_resultados.merge(calcular_proporciones_y_totales(df, cols_macro, 'RUT', 'sect_rut_macrozona', 'total_rut_macrozona'), on=cols_macro, how='left')
df_resultados = df_resultados.merge(calcular_proporciones_y_totales(df, cols_macro, 'CLAVES', 'sect_claves_macrozona', 'total_claves_macrozona'), on=cols_macro, how='left')
# Actualizamos el nombre de la columna a _mwh
df_resultados = df_resultados.merge(calcular_proporciones_y_totales(df, cols_macro, 'ENERGIA', 'sect_energia_macrozona', 'total_energia_macrozona_mwh'), on=cols_macro, how='left')
df_resultados = df_resultados.merge(calcular_proporciones_y_totales(df, cols_macro, 'CALOR', 'sect_calor_macrozona', 'total_calor_macrozona_mwh'), on=cols_macro, how='left')

In [5]:
# ==========================================
# CÁLCULO DEL FACTOR DE FORMA POR CLAVE Y CLUSTER
# ==========================================

# Paso A: Agrupar por clave_unica para sumar las 24 horas y encontrar la mínima (el mayor retiro)
# Mantenemos macrozona e id_cluster en el groupby para no perderlos
df_factor = df.groupby(['macrozona', 'id_cluster', 'clave_unica']).agg(
    suma_diaria=('medida_mean', 'sum'),
    min_horario=('medida_mean', 'min')
).reset_index()

# Paso B: Aplicar la fórmula
# Usamos np.where para evitar que el programa falle dividiendo por cero 
# si alguna clave_unica no tuvo consumo ese día (min_horario == 0)
df_factor['factor_forma'] = np.where(
    df_factor['min_horario'] == 0,
    0,  # Si no hay consumo, asumimos un factor de 0
    1 - (df_factor['suma_diaria'] / (df_factor['min_horario'] * 24))
)

# Paso C: Calcular el factor de forma promedio de cada cluster
factor_promedio_cluster = df_factor.groupby(['macrozona', 'id_cluster'])['factor_forma'].mean().reset_index(name='factor_forma_promedio')

# Opcional: También puedes sacar la desviación estándar para ver si los perfiles dentro del cluster se comportan igual
factor_std_cluster = df_factor.groupby(['macrozona', 'id_cluster'])['factor_forma'].std().fillna(0).reset_index(name='factor_forma_std')

# Paso D: Unir esta nueva métrica a tu tabla maestra (df_resultados)
df_resultados = df_resultados.merge(factor_promedio_cluster, on=['macrozona', 'id_cluster'], how='left')
df_resultados = df_resultados.merge(factor_std_cluster, on=['macrozona', 'id_cluster'], how='left')

# Paso E: Mapear al DataFrame original
# Pegamos el cálculo directamente en 'df' para que cada fila horaria sepa cuál es su factor
df['factor_forma'] = df['clave_unica'].map(df_factor.set_index('clave_unica')['factor_forma'])

In [6]:
# ==========================================
# CÁLCULO DE LA CAPACIDAD DE CARGA Y EMPALME
# ==========================================

# Paso A: Agrupar por clave_unica para extraer los datos base
# - Tomamos el mínimo absoluto de 'medida_min' (el mayor retiro de 15 min)
# - Tomamos 'first' para medida_total y medida_count ya que se repiten por clave
df_carga = df.groupby(['macrozona', 'id_cluster', 'clave_unica']).agg(
    peak_15min=('medida_min', 'min'),
    m_total=('medida_total', 'first'),
    m_count=('medida_count', 'first')
).reset_index()

# Paso B: Calcular los componentes
# 1. Potencia máxima (Como peak_15min es Energía, al dividir por 0.25h obtenemos Potencia en kW)
df_carga['potencia_max_horaria'] = df_carga['peak_15min'] / 0.25

# 2. Energía máxima teórica (Potencia Peak * Total de horas del periodo)
df_carga['energia_max_teorica'] = df_carga['potencia_max_horaria'] * (df_carga['m_count'] * 24)

# 3. Energía real del periodo
df_carga['energia_anual_real'] = df_carga['m_total'] * df_carga['m_count']

# Paso C: Calcular la capacidad de carga (Load Factor directo)
df_carga['capacidad_carga'] = np.where(
    df_carga['energia_max_teorica'] == 0,
    0,
    df_carga['energia_anual_real'] / df_carga['energia_max_teorica']
)

# Paso D: Calcular los promedios por Cluster para la tabla maestra
# Agregamos el cálculo de la potencia máxima promedio (capacidad de empalme)
capacidad_promedio_cluster = df_carga.groupby(['macrozona', 'id_cluster'])['capacidad_carga'].mean().reset_index(name='capacidad_carga_promedio')
capacidad_std_cluster = df_carga.groupby(['macrozona', 'id_cluster'])['capacidad_carga'].std().fillna(0).reset_index(name='capacidad_carga_std')
potencia_max_promedio_cluster = df_carga.groupby(['macrozona', 'id_cluster'])['potencia_max_horaria'].mean().reset_index(name='potencia_max_promedio_kw')

# Paso E: Unir a la tabla de resultados maestra (df_resultados)
df_resultados = df_resultados.merge(capacidad_promedio_cluster, on=['macrozona', 'id_cluster'], how='left')
df_resultados = df_resultados.merge(capacidad_std_cluster, on=['macrozona', 'id_cluster'], how='left')
df_resultados = df_resultados.merge(potencia_max_promedio_cluster, on=['macrozona', 'id_cluster'], how='left')

# Paso F: Mapear al DataFrame original (df)
# Traemos tanto la capacidad_carga como la potencia_max_horaria individual
df = df.merge(df_carga[['clave_unica', 'capacidad_carga', 'potencia_max_horaria']], on='clave_unica', how='left')

# ==========================================
# VISTAS RÁPIDAS
# ==========================================
print("--- DETALLE DE CAPACIDAD DE CARGA Y EMPALME POR CLUSTER ---")
columnas_ver = ['macrozona', 'id_cluster', 'capacidad_carga_promedio', 'potencia_max_promedio_kw']
print(df_resultados[columnas_ver].head())

--- DETALLE DE CAPACIDAD DE CARGA Y EMPALME POR CLUSTER ---
  macrozona  id_cluster  capacidad_carga_promedio  potencia_max_promedio_kw
0    Centro           0                  0.426641              -2288.523601
1    Centro           1                  0.193437               -801.506871
2    Centro           2                  0.298861               -745.481131
3    Centro           3                  0.404540              -1986.106775
4    Centro           4                  0.527077              -2785.366539


In [7]:
df.columns

Index(['clave', 'RUT_CLIENTE', 'REGION_CLIENTE', 'macrozona', 'Zona', 'Hora',
       'medida_count', 'medida_min', 'CLIENTE', 'CLIENTE_log', 'n_clientes',
       'TIPO', 'NOMBRE_ESTABLECIMIENTO', 'COMBUSTIBLE_PRIMARIO', 'SECTOR',
       'SUBSECTOR', 'RUBRO', 'DEMANDA_CALOR_MWH_sum', 'DEMANDA_CALOR_MWH_mean',
       'DEMANDA_CALOR_MWH_std', 'DEMANDA_CALOR_MWH_max',
       'DEMANDA_CALOR_MWH_min', 'RUT_PROVEEDOR', 'RUT_PROVEEDOR_log',
       'n_rut_proveedores', 'PROVEEDOR', 'PROVEEDOR_log', 'n_proveedores',
       'nombre_barra', 'nombre_barra_log', 'n_nombres_barra', 'tension',
       'tension_log', 'n_tensiones', 'Nombre_Corto', 'Nombre_Corto_log',
       'n_nombres_cortos', 'periodo_last', 'periodos_log', 'meses_operados',
       'medida_mean', 'medida_std', 'CMg[CLP/KWh]_mean', 'CMg[CLP/KWh]_std',
       'CMg[CLP/KWh]_count', 'valorizado_CLP_mean', 'valorizado_CLP_std',
       'valorizado_CLP_count', 'medida_total', 'medida_porcentual',
       'clave_unica', 'id_cluster', 'metodo_cl

In [8]:
# =========================================
# PARÁMETROS GLOBALES
# ==========================================
eficiencia_conversion = 0.90
precio_dolar = 926.41
mwh_bateria = 2

# ==========================================
# 1. CÁLCULO DE BATERÍAS REQUERIDAS
# ==========================================
df["n_baterias_requeridas"] = np.ceil(df["DEMANDA_CALOR_MWH_sum"] / 365 / mwh_bateria)


# ==========================================
# 2. HOLGURA ELÉCTRICA Y FILTRO HORARIO
# ==========================================
df["diff_max"] = (df["medida_mean"] - df["potencia_max_horaria"]).clip(lower=0)
df_solar = df[df["Hora"].isin([10,11,12,13,14,15,16,17])].copy()

df_solar["costo_x_diff"] = df_solar["CMg[CLP/KWh]_mean"] * df_solar["diff_max"]


# ==========================================
# 3. AGRUPACIÓN Y PROMEDIO PONDERADO
# ==========================================
df_solar_mean = df_solar.groupby("clave_unica").agg({
    "costo_x_diff": "sum",
    "diff_max": "sum"
}).reset_index()

df_solar_mean["costo_marginal_mean_solar"] = (df_solar_mean["costo_x_diff"] / df_solar_mean["diff_max"]).fillna(0)
df_solar_mean.drop(columns=["costo_x_diff"], inplace=True)


# ==========================================
# 4. CÁLCULOS DE CARGA Y COSTOS
# ==========================================
energia_electrica_necesaria_kwh = (mwh_bateria * 1000) / eficiencia_conversion

# Baterías y costos totales en CLP
df_solar_mean["n_baterias_cargadas"] = np.floor(df_solar_mean["diff_max"] / energia_electrica_necesaria_kwh)
df_solar_mean["costo_carga_bateria_unitaria"] = df_solar_mean["costo_marginal_mean_solar"] * energia_electrica_necesaria_kwh
df_solar_mean["costo_carga_bateria_total"] = df_solar_mean["costo_carga_bateria_unitaria"] * df_solar_mean["n_baterias_cargadas"]

# Costo térmico (CLP/MWht) con protección NaN
df_solar_mean["costo_MWht_bateria"] = np.where(
    df_solar_mean["n_baterias_cargadas"] > 0,
    df_solar_mean["costo_carga_bateria_total"] / (df_solar_mean["n_baterias_cargadas"] * mwh_bateria),
    np.nan
)

# NUEVO: Conversión directa a USD/MWht en este paso
df_solar_mean["costo_usd_mwht"] = df_solar_mean["costo_MWht_bateria"] / precio_dolar


# ==========================================
# 5. FUSIÓN DE DATOS (NIVEL INSTALACIÓN)
# ==========================================
# Borramos diff_max si existe para evitar duplicados en el merge
if "diff_max" in df.columns:
    df.drop(columns=["diff_max"], inplace=True)

columnas_a_fusionar = [
    "clave_unica", 
    "costo_marginal_mean_solar", 
    "n_baterias_cargadas", 
    "costo_carga_bateria_unitaria", 
    "costo_carga_bateria_total",
    "costo_MWht_bateria",
    "costo_usd_mwht" # Se fusiona la columna en dólares
]

df = df.merge(df_solar_mean[columnas_a_fusionar], on="clave_unica", how="left")
df["costo_marginal_mean_solar"] = df["costo_marginal_mean_solar"].fillna(0)
df["mwh_bateria"] = mwh_bateria


# ==========================================
# 6. CÁLCULO DE MERCADO FACTIBLE Y REGISTRO EN DF (NIVEL RUT)
# ==========================================
# 6.1. Baterías Cargables (Oferta total del cliente)
baterias_cargables_por_rut = df.drop_duplicates(
    subset=["RUT_CLIENTE", "REGION_CLIENTE", "clave_unica", "macrozona"]
).groupby(["RUT_CLIENTE", "REGION_CLIENTE", "macrozona"]).agg(
    baterias_cargables_por_rut=('n_baterias_cargadas', 'sum')
).reset_index()

# 6.2. Baterías Requeridas (Demanda del cliente)
baterias_requeridas_por_rut = df.drop_duplicates(
    subset=["RUT_CLIENTE", "REGION_CLIENTE", "macrozona"]
).groupby(["RUT_CLIENTE", "REGION_CLIENTE", "macrozona"]).agg(
    baterias_requeridas_por_rut=('n_baterias_requeridas', 'first') 
).reset_index()

# 6.3. Cruce de datos por RUT
mercado_por_rut = pd.merge(
    baterias_cargables_por_rut, 
    baterias_requeridas_por_rut, 
    on=["RUT_CLIENTE", "REGION_CLIENTE", "macrozona"], 
    how="outer"
).fillna(0)

# 6.4. Cálculo del mínimo (restricción de mercado)
mercado_por_rut["demanda_baterias_factible"] = mercado_por_rut[["baterias_cargables_por_rut", "baterias_requeridas_por_rut"]].min(axis=1)

# 6.5. INTEGRACIÓN FINAL AL DF ORIGINAL
columnas_mercado = [
    "RUT_CLIENTE", "REGION_CLIENTE", "macrozona", 
    "baterias_cargables_por_rut", 
    "baterias_requeridas_por_rut", 
    "demanda_baterias_factible"
]

df = df.merge(mercado_por_rut[columnas_mercado], on=["RUT_CLIENTE", "REGION_CLIENTE", "macrozona"], how="left")

In [9]:
calor = pd.read_csv(Path(r"D:\ProyectoAnalisisElectrico\CupraThermV2\Resumen_Establecimientos.csv"), sep=",")


In [10]:
cols = ["clave", "n_baterias_cargadas", "RUT_CLIENTE", "REGION_CLIENTE", "macrozona", "id_cluster",
                                 "demanda_baterias_factible", "costo_usd_mwht", "CLIENTE", "mwh_bateria"]
calor_interes = pd.merge(df[(df["demanda_baterias_factible"] > 0)&(df["n_baterias_cargadas"] > 0)].drop_duplicates(subset=cols)[cols], 
                    calor[["NOMBRE_ESTABLECIMIENTO", "COMBUSTIBLE_PRIMARIO", "DEMANDA_CALOR_MWH", "REGION", "RUT_RAZON_SOCIAL"]], 
                    left_on=["RUT_CLIENTE", "REGION_CLIENTE"], right_on=["RUT_RAZON_SOCIAL", "REGION"], how="left")

# Diccionario de costos homologados en USD/MWht para los valores únicos de calor_interes
precios_combustibles = {
    'Petróleo N 2 (Diesel)': 101.73,
    'MDO (Marine Diesel Oil)': 101.73,
    'Gas Licuado de Petróleo': 66.57,
    'Propano': 66.57,
    'Petróleo N 6': 64.47,
    'Petróleo N 5': 64.47,
    'Gas Natural': 32.46,
    'Carbón Bituminoso': 18.69,
    'Coke de Petróleo (Petcoke)': 18.69,
    'Varios': 18.69  # Criterio conservador
}

# Mapeamos los costos al DataFrame calor_interes
calor_interes['costo_actual_usd_mwht'] = calor_interes['COMBUSTIBLE_PRIMARIO'].map(precios_combustibles)



In [11]:
# 1. Asegurar que tenemos el total de baterías por grupo
calor_interes['total_baterias_cargadas'] = (
    calor_interes.groupby(['RUT_CLIENTE', 'REGION_CLIENTE'])['n_baterias_cargadas']
    .transform('sum')
)

# 2. Definir la función para calcular el promedio ponderado por grupo
def calc_promedio_ponderado(group):
    pesos = (
        (group['n_baterias_cargadas'] / group['total_baterias_cargadas']) 
        * group['demanda_baterias_factible'] 
        * group['costo_usd_mwht']
    )
    if pesos.sum() == 0:
        return np.nan
    return np.average(group['costo_usd_mwht'], weights=pesos)

# 3. Aplicar la función agrupando por RUT y Región
df_resumen = (
    calor_interes.groupby(['RUT_CLIENTE', 'REGION_CLIENTE'])
    .apply(calc_promedio_ponderado)
    .reset_index(name='promedio_ponderado_costo_mwht_bateria')
)

# Combinar el resumen con el DataFrame original usando las llaves de agrupación
calor_interes = calor_interes.merge(
    df_resumen, 
    on=['RUT_CLIENTE', 'REGION_CLIENTE'], 
    how='left'
)

In [12]:
# 1. Crear un DataFrame temporal solo con datos únicos por Establecimiento
# ¡Agregamos las columnas de la batería para saber su límite!
columnas_unicas = [
    'RUT_CLIENTE', 'REGION_CLIENTE', 'NOMBRE_ESTABLECIMIENTO', 
    'DEMANDA_CALOR_MWH', 'costo_actual_usd_mwht', 'promedio_ponderado_costo_mwht_bateria',
    'total_baterias_cargadas', 'mwh_bateria' 
]
df_establecimientos = calor_interes[columnas_unicas].drop_duplicates().copy()

# 2. Cálculos base en esta vista sin duplicados
# Costo por calor para el promedio ponderado
df_establecimientos['costo_por_calor'] = (
    df_establecimientos['DEMANDA_CALOR_MWH'] * df_establecimientos['costo_actual_usd_mwht']
)

# Convertir demanda de calor ANUAL a DIARIA
df_establecimientos['demanda_calor_diaria'] = df_establecimientos['DEMANDA_CALOR_MWH'] / 365

# 3. Agrupar por RUT y Región para sumar y consolidar
agrupado = df_establecimientos.groupby(['RUT_CLIENTE', 'REGION_CLIENTE']).agg(
    suma_calor_total_anual=('DEMANDA_CALOR_MWH', 'sum'),
    suma_calor_total_diaria=('demanda_calor_diaria', 'sum'),
    suma_costo_por_calor=('costo_por_calor', 'sum'),
    costo_bateria=('promedio_ponderado_costo_mwht_bateria', 'first'),
    baterias_totales=('total_baterias_cargadas', 'first'),
    mwh_por_bateria=('mwh_bateria', 'first')
).reset_index()

# 4. Calcular el promedio ponderado, la cobertura real y el ahorro
# Promedio ponderado del combustible
agrupado['promedio_ponderado_costo_mwht_combustible'] = (
    agrupado['suma_costo_por_calor'] / agrupado['suma_calor_total_anual']
)

# Calcular cuánta energía DIARIA entregan las baterías en total para este RUT/Región
agrupado['energia_bateria_diaria_disponible'] = agrupado['baterias_totales'] * agrupado['mwh_por_bateria']

# El calor cubierto será el mínimo entre lo que piden (demanda diaria) y lo que hay (batería)
agrupado['calor_diario_cubierto'] = np.minimum(
    agrupado['suma_calor_total_diaria'], 
    agrupado['energia_bateria_diaria_disponible']
)

# Ahorro Total: (Calor diario cubierto * 365 para anualizar) * (Diferencia de costos)
agrupado['ahorro_total_rut_region_usd'] = (agrupado['calor_diario_cubierto'] * 365) * (
    agrupado['promedio_ponderado_costo_mwht_combustible'] - agrupado['costo_bateria']
)

# 5. Llevar los resultados finales de vuelta al DataFrame original
columnas_finales = [
    'RUT_CLIENTE', 'REGION_CLIENTE', 
    'promedio_ponderado_costo_mwht_combustible', 'ahorro_total_rut_region_usd'
]

# Cruzamos la información original con los cálculos limpios
calor_interes = calor_interes.merge(
    agrupado[columnas_finales], 
    on=['RUT_CLIENTE', 'REGION_CLIENTE'], 
    how='left'
)

In [13]:
# 1. Crear la vista única sin claves duplicadas
columnas_unicas = [
    'RUT_CLIENTE', 'REGION_CLIENTE', 'NOMBRE_ESTABLECIMIENTO', 
    'DEMANDA_CALOR_MWH', 'costo_actual_usd_mwht', 
    'promedio_ponderado_costo_mwht_bateria', 'total_baterias_cargadas', 'mwh_bateria'
]
df_est = calor_interes[columnas_unicas].drop_duplicates().copy()

# 2. ESTANDARIZAR A ESCALA DIARIA
df_est['demanda_calor_diaria'] = df_est['DEMANDA_CALOR_MWH'] / 365
# Asumimos que esta fórmula ya representa la energía DIARIA que entrega la batería
df_est['energia_bateria_diaria_disponible'] = df_est['total_baterias_cargadas'] * df_est['mwh_bateria']

# 3. ORDENAR: Del combustible más caro al más barato
df_est = df_est.sort_values(
    by=['RUT_CLIENTE', 'REGION_CLIENTE', 'costo_actual_usd_mwht'], 
    ascending=[True, True, False]
)

# 4. Suma acumulada de la demanda DIARIA
df_est['demanda_diaria_acumulada'] = df_est.groupby(['RUT_CLIENTE', 'REGION_CLIENTE'])['demanda_calor_diaria'].cumsum()
df_est['demanda_diaria_acumulada_previa'] = df_est['demanda_diaria_acumulada'] - df_est['demanda_calor_diaria']

# 5. Calcular cuánto calor DIARIO se reemplaza realmente en cada fila
df_est['bateria_diaria_restante'] = np.maximum(0, df_est['energia_bateria_diaria_disponible'] - df_est['demanda_diaria_acumulada_previa'])

# Reemplazo diario: el mínimo entre lo que pide el local hoy y la batería que queda hoy
df_est['calor_diario_reemplazado'] = np.minimum(df_est['demanda_calor_diaria'], df_est['bateria_diaria_restante'])

# 6. Calcular el ahorro ANUALIZADO por establecimiento
# Multiplicamos el calor diario reemplazado por 365 para volver a llevarlo al año, y luego por la diferencia de costos
df_est['ahorro_anual_establecimiento'] = (df_est['calor_diario_reemplazado'] * 365) * (
    df_est['costo_actual_usd_mwht'] - df_est['promedio_ponderado_costo_mwht_bateria']
)

# 7. Agrupar el ahorro total anual y cruzarlo con el DataFrame original
ahorro_total = df_est.groupby(['RUT_CLIENTE', 'REGION_CLIENTE'])['ahorro_anual_establecimiento'].sum().reset_index(name='ahorro_total_optimo_anual_usd')

calor_interes = calor_interes.merge(
    ahorro_total, 
    on=['RUT_CLIENTE', 'REGION_CLIENTE'], 
    how='left'
)

In [14]:
# 1. Definir las llaves de cruce y las columnas que te interesan
columnas_metricas = [
    'RUT_CLIENTE', 
    'REGION_CLIENTE', 
    'macrozona', # La incluimos por si tu df la usa como llave
    'promedio_ponderado_costo_mwht_bateria', 
    'promedio_ponderado_costo_mwht_combustible', 
    'ahorro_total_rut_region_usd', 
    'ahorro_total_optimo_anual_usd'
]

# 2. Crear una vista limpia sin duplicados desde calor_interes
vista_costos_ahorros = calor_interes[columnas_metricas].drop_duplicates(
    subset=['RUT_CLIENTE', 'REGION_CLIENTE']
)

# 3. (Opcional pero recomendado) Limpiar 'df' por si alguna de estas columnas ya existía de pruebas anteriores
columnas_nuevas = [
    'promedio_ponderado_costo_mwht_bateria', 
    'promedio_ponderado_costo_mwht_combustible', 
    'ahorro_total_rut_region_usd', 
    'ahorro_total_optimo_anual_usd'
]
df = df.drop(columns=[col for col in columnas_nuevas if col in df.columns], errors='ignore')

# 4. Cruzar la información hacia tu DataFrame maestro 'df'
df = df.merge(
    vista_costos_ahorros, 
    on=['RUT_CLIENTE', 'REGION_CLIENTE', 'macrozona'], 
    how='left'
)

In [15]:
df

,clave,RUT_CLIENTE,REGION_CLIENTE,macrozona,Zona,Hora,medida_count,medida_min,CLIENTE,CLIENTE_log,...,costo_MWht_bateria,costo_usd_mwht,mwh_bateria,baterias_cargables_por_rut,baterias_requeridas_por_rut,demanda_baterias_factible,promedio_ponderado_costo_mwht_bateria,promedio_ponderado_costo_mwht_combustible,ahorro_total_rut_region_usd,ahorro_total_optimo_anual_usd
0,$C$439,79587210-8,Antofagasta,Norte Grande,Norte,0,365,-6463.228844,MINERA ESCONDIDA LIMITADA,MINERA ESCONDIDA LIMITADA,...,6549.734777,7.070017,2,1730.0,180.0,180.0,6.735354,101.73,1.242869e+07,1.242869e+07
1,$C$439,79587210-8,Antofagasta,Norte Grande,Norte,1,365,-6541.272978,MINERA ESCONDIDA LIMITADA,MINERA ESCONDIDA LIMITADA,...,6549.734777,7.070017,2,1730.0,180.0,180.0,6.735354,101.73,1.242869e+07,1.242869e+07
2,$C$439,79587210-8,Antofagasta,Norte Grande,Norte,2,365,-6307.120603,MINERA ESCONDIDA LIMITADA,MINERA ESCONDIDA LIMITADA,...,6549.734777,7.070017,2,1730.0,180.0,180.0,6.735354,101.73,1.242869e+07,1.242869e+07
3,$C$439,79587210-8,Antofagasta,Norte Grande,Norte,3,365,-6495.837207,MINERA ESCONDIDA LIMITADA,MINERA ESCONDIDA LIMITADA,...,6549.734777,7.070017,2,1730.0,180.0,180.0,6.735354,101.73,1.242869e+07,1.242869e+07
4,$C$439,79587210-8,Antofagasta,Norte Grande,Norte,4,365,-6507.573554,MINERA ESCONDIDA LIMITADA,MINERA ESCONDIDA LIMITADA,...,6549.734777,7.070017,2,1730.0,180.0,180.0,6.735354,101.73,1.242869e+07,1.242869e+07
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14971,VENTPTACOBRE,96561560-1,Atacama,Norte Chico,Norte Distribución,19,365,-205.923660,SOCIEDAD PUNTA DEL COBRE S.A.,SOCIEDAD PUNTA DEL COBRE S.A.,...,7181.475317,7.751941,2,32.0,3.0,3.0,7.706734,101.73,1.693923e+05,1.693923e+05
14972,VENTPTACOBRE,96561560-1,Atacama,Norte Chico,Norte Distribución,20,365,-205.354553,SOCIEDAD PUNTA DEL COBRE S.A.,SOCIEDAD PUNTA DEL COBRE S.A.,...,7181.475317,7.751941,2,32.0,3.0,3.0,7.706734,101.73,1.693923e+05,1.693923e+05
14973,VENTPTACOBRE,96561560-1,Atacama,Norte Chico,Norte Distribución,21,365,-205.323410,SOCIEDAD PUNTA DEL COBRE S.A.,SOCIEDAD PUNTA DEL COBRE S.A.,...,7181.475317,7.751941,2,32.0,3.0,3.0,7.706734,101.73,1.693923e+05,1.693923e+05
14974,VENTPTACOBRE,96561560-1,Atacama,Norte Chico,Norte Distribución,22,365,-204.061630,SOCIEDAD PUNTA DEL COBRE S.A.,SOCIEDAD PUNTA DEL COBRE S.A.,...,7181.475317,7.751941,2,32.0,3.0,3.0,7.706734,101.73,1.693923e+05,1.693923e+05


In [16]:
# 2. Agrupar para obtener el valor máximo de baterías por cada 'clave' única dentro de un RUT+REGION
df_claves = df.groupby(['RUT_CLIENTE', 'REGION_CLIENTE', 'clave'], as_index=False)['n_baterias_cargadas'].max()

# 3. Ordenar de mayor a menor y extraer solo el top 3 por cada grupo
df_top = df_claves.sort_values(
    by=['RUT_CLIENTE', 'REGION_CLIENTE', 'n_baterias_cargadas'], 
    ascending=[True, True, False]
)
df_top = df_top.groupby(['RUT_CLIENTE', 'REGION_CLIENTE']).head(3).copy()

# 4. Crear una columna de ranking (1, 2, 3)
df_top['ranking'] = df_top.groupby(['RUT_CLIENTE', 'REGION_CLIENTE']).cumcount() + 1

# 5. Pivotear la tabla para convertir las filas del top en columnas
df_pivot = df_top.pivot(
    index=['RUT_CLIENTE', 'REGION_CLIENTE'], 
    columns='ranking', 
    values=['clave', 'n_baterias_cargadas']
)

# 6. Renombrar las columnas al formato solicitado
nuevas_columnas = []
for col_name, rank in df_pivot.columns:
    if col_name == 'clave':
        nuevas_columnas.append(f'clave_top_{rank}')
    else:
        nuevas_columnas.append(f'n_clave_top_{rank}')
        
df_pivot.columns = nuevas_columnas
df_pivot = df_pivot.reset_index()

# 7. Sobrescribir el DataFrame original
df = df.merge(df_pivot, on=['RUT_CLIENTE', 'REGION_CLIENTE'], how='left')

In [17]:
df.head()

,clave,RUT_CLIENTE,REGION_CLIENTE,macrozona,Zona,Hora,medida_count,medida_min,CLIENTE,CLIENTE_log,...,promedio_ponderado_costo_mwht_bateria,promedio_ponderado_costo_mwht_combustible,ahorro_total_rut_region_usd,ahorro_total_optimo_anual_usd,clave_top_1,clave_top_2,clave_top_3,n_clave_top_1,n_clave_top_2,n_clave_top_3
0,$C$439,79587210-8,Antofagasta,Norte Grande,Norte,0,365,-6463.228844,MINERA ESCONDIDA LIMITADA,MINERA ESCONDIDA LIMITADA,...,6.735354,101.73,1.242869e+07,1.242869e+07,PURI_COLB,PURI_ENEL,FARE_ENEL,209.0,193.0,140.0
1,$C$439,79587210-8,Antofagasta,Norte Grande,Norte,1,365,-6541.272978,MINERA ESCONDIDA LIMITADA,MINERA ESCONDIDA LIMITADA,...,6.735354,101.73,1.242869e+07,1.242869e+07,PURI_COLB,PURI_ENEL,FARE_ENEL,209.0,193.0,140.0
2,$C$439,79587210-8,Antofagasta,Norte Grande,Norte,2,365,-6307.120603,MINERA ESCONDIDA LIMITADA,MINERA ESCONDIDA LIMITADA,...,6.735354,101.73,1.242869e+07,1.242869e+07,PURI_COLB,PURI_ENEL,FARE_ENEL,209.0,193.0,140.0
3,$C$439,79587210-8,Antofagasta,Norte Grande,Norte,3,365,-6495.837207,MINERA ESCONDIDA LIMITADA,MINERA ESCONDIDA LIMITADA,...,6.735354,101.73,1.242869e+07,1.242869e+07,PURI_COLB,PURI_ENEL,FARE_ENEL,209.0,193.0,140.0
4,$C$439,79587210-8,Antofagasta,Norte Grande,Norte,4,365,-6507.573554,MINERA ESCONDIDA LIMITADA,MINERA ESCONDIDA LIMITADA,...,6.735354,101.73,1.242869e+07,1.242869e+07,PURI_COLB,PURI_ENEL,FARE_ENEL,209.0,193.0,140.0


In [18]:
df.to_parquet(carpeta_datos / "2505_2604_clustered_porcentual_actives_processed.parquet", index=False)

df_resultados['lista_ruts'] = df_resultados['lista_ruts'].apply(lambda x: "::".join(map(str, x)))
df_resultados['lista_nombres'] = df_resultados['lista_nombres'].apply(lambda x: "::".join(map(str, x)))

df_resultados.to_parquet(carpeta_datos / "2505_2604_clustered_porcentual_actives_resultados.parquet", index=False)

In [19]:
df.columns

Index(['clave', 'RUT_CLIENTE', 'REGION_CLIENTE', 'macrozona', 'Zona', 'Hora',
       'medida_count', 'medida_min', 'CLIENTE', 'CLIENTE_log', 'n_clientes',
       'TIPO', 'NOMBRE_ESTABLECIMIENTO', 'COMBUSTIBLE_PRIMARIO', 'SECTOR',
       'SUBSECTOR', 'RUBRO', 'DEMANDA_CALOR_MWH_sum', 'DEMANDA_CALOR_MWH_mean',
       'DEMANDA_CALOR_MWH_std', 'DEMANDA_CALOR_MWH_max',
       'DEMANDA_CALOR_MWH_min', 'RUT_PROVEEDOR', 'RUT_PROVEEDOR_log',
       'n_rut_proveedores', 'PROVEEDOR', 'PROVEEDOR_log', 'n_proveedores',
       'nombre_barra', 'nombre_barra_log', 'n_nombres_barra', 'tension',
       'tension_log', 'n_tensiones', 'Nombre_Corto', 'Nombre_Corto_log',
       'n_nombres_cortos', 'periodo_last', 'periodos_log', 'meses_operados',
       'medida_mean', 'medida_std', 'CMg[CLP/KWh]_mean', 'CMg[CLP/KWh]_std',
       'CMg[CLP/KWh]_count', 'valorizado_CLP_mean', 'valorizado_CLP_std',
       'valorizado_CLP_count', 'medida_total', 'medida_porcentual',
       'clave_unica', 'id_cluster', 'metodo_cl